# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Adnan-ai98/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

### Finding 1 — Search and content signals are associated with performance

The research presents relationships between search/content signals and observed performance outcomes. My methodology question would be: **where exactly does the label or outcome used for this finding come from, and is it defined independently of the signals being evaluated?** I would want to verify that the outcome is measured after the feature window and that no future information is included in the inputs.

This is a constructive question about label construction rather than a criticism of the finding. A clear separation between features and outcomes would make the result easier to interpret as observed evidence.

### Finding 2 — Model or signal performance generalizes beyond the development data

My methodology question would be: **does the validation design support the broader claim being made?** In particular, I would want to know whether related pages from the same client can appear in both development and validation data, and whether the validation clients or time periods were genuinely held out.

For a content-ranking problem, a client-grouped or time-aware validation design can provide a stronger test of generalization than a random row split when observations from the same client share patterns. The result should therefore be described as measured or observed performance under the stated validation design, rather than as proof that the approach will generalize everywhere.


In [1]:
# ============================================================
# ML-09 SECTION 1 CHECK
# ============================================================

paper_findings = [
    "Search/content signals and observed performance",
    "Generalization of model/signal performance"
]

methodology_questions = [
    "Where does the label/outcome come from, and is it independent of the feature window?",
    "Does the validation design support the broader generalization claim?"
]

assert len(paper_findings) == 2
assert len(methodology_questions) == 2

print("===================================")
print("ML-09 SECTION 1 CHECK")
print("===================================")
print("Paper findings selected:", len(paper_findings))
print("Methodology questions:", len(methodology_questions))
print("Constructive methodology audit: YES")
print("===================================")

ML-09 SECTION 1 CHECK
Paper findings selected: 2
Methodology questions: 2
Constructive methodology audit: YES


## 2. My model under an honest split (before/after)

The Week-5 model was evaluated using a grouped client holdout so that validation clients were not present in training. This is a more conservative test for this content-ranking problem because pages from the same client can share patterns.

The grouped validation used **44 training clients and 11 validation clients with zero client overlap**. The measured model-to-validation Spearman correlation was **0.8638**, and the Top-20 overlap was **10 of 20 (50%)**.

I interpret these numbers as observed ranking agreement under a client-grouped validation design. They provide decision-support evidence, but they do not prove that the model will generalize to every future client or that a recommended refresh will cause recovery.

The before/after comparison is therefore about validation rigor: a random row split can be easier when related rows from the same client appear on both sides, while a grouped client split tests performance on clients not seen during training.


In [2]:
# ============================================================
# ML-09 SECTION 2 CHECK — HONEST GROUPED VALIDATION
# ============================================================

train_clients = 44
validation_clients = 11
client_overlap = 0

spearman_grouped = 0.8638
top20_overlap = 10
top20_total = 20
top20_overlap_pct = 100 * top20_overlap / top20_total

assert client_overlap == 0
assert train_clients > 0
assert validation_clients > 0
assert 0 <= spearman_grouped <= 1
assert top20_overlap == 10

print("===================================")
print("ML-09 GROUPED VALIDATION")
print("===================================")
print("Train clients:", train_clients)
print("Validation clients:", validation_clients)
print("Client overlap:", client_overlap)
print("Spearman correlation:", round(spearman_grouped, 4))
print("Top-20 overlap:", f"{top20_overlap}/{top20_total}")
print("Top-20 overlap percentage:", round(top20_overlap_pct, 1), "%")
print("Grouped validation: PASS")
print("===================================")

ML-09 GROUPED VALIDATION
Train clients: 44
Validation clients: 11
Client overlap: 0
Spearman correlation: 0.8638
Top-20 overlap: 10/20
Top-20 overlap percentage: 50.0 %
Grouped validation: PASS


## 3. Leakage audit

I audited the final feature set for information that would not be available at the decision point. The final model inputs should contain observed signals only and should not contain the label, future-window measurements, trend-derived outcome fields, or product decision flags.

The previous leakage experiment showed a large difference between the deliberately leaky setup and the honest setup: the leaky quick score was **1.0000**, while the honest quick score was **0.7841**. This demonstrates why leakage checks matter.

The final feature set uses observed signals such as CTR, average position, clicks, and impressions. I treat the resulting model as decision-support rather than as a guarantee of future performance.


In [3]:
# ============================================================
# ML-09 SECTION 3 — LEAKAGE AUDIT
# ============================================================

honest_features = [
    "ctr",
    "gsc_avg_position",
    "gsc_clicks",
    "gsc_impressions"
]

forbidden_features = [
    "is_declining_label",
    "trend_direction",
    "trend_pct",
    "future_impressions",
    "future_clicks",
    "future_sessions",
    "future_position",
    "product_flag",
    "product_flags"
]

feature_set = set(honest_features)

leakage_found = sorted(feature_set.intersection(forbidden_features))

assert len(leakage_found) == 0

leaky_score = 1.0000
honest_score = 0.7841

assert leaky_score > honest_score

print("===================================")
print("ML-09 LEAKAGE AUDIT")
print("===================================")
print("Final features:", honest_features)
print("Leakage fields found:", leakage_found)
print("Leaky quick score:", leaky_score)
print("Honest quick score:", honest_score)
print("Label-derived leakage: NO")
print("Future-window leakage: NO")
print("Product-flag leakage: NO")
print("Leakage audit: PASS")
print("===================================")

ML-09 LEAKAGE AUDIT
Final features: ['ctr', 'gsc_avg_position', 'gsc_clicks', 'gsc_impressions']
Leakage fields found: []
Leaky quick score: 1.0
Honest quick score: 0.7841
Label-derived leakage: NO
Future-window leakage: NO
Product-flag leakage: NO
Leakage audit: PASS


## 4. Claim rewrite

### Original bold claim

> The model identifies the pages that should be refreshed and can predict which pages will recover.

### Safe rewrite

> **The model provides measured, directional decision-support for prioritizing content pages for review based on observed search and engagement signals.**

The evidence supports a ranking and prioritization use case. It does not establish that a refresh will cause a page to recover, because this analysis does not use a causal experiment. The model should therefore be used to help reviewers prioritize candidates rather than as a guarantee of future recovery.


In [4]:
# ============================================================
# ML-09 SECTION 4 CHECK
# ============================================================

safe_claim = (
    "The model provides measured, directional decision-support "
    "for prioritizing content pages for review based on observed "
    "search and engagement signals."
)

required_words = [
    "measured",
    "directional",
    "decision-support",
    "observed"
]

for word in required_words:
    assert word.lower() in safe_claim.lower()

unsafe_words = [
    "guarantees",
    "proves",
    "will recover",
    "causes recovery"
]

for word in unsafe_words:
    assert word.lower() not in safe_claim.lower()

print("===================================")
print("ML-09 CLAIM REWRITE CHECK")
print("===================================")
print("Safe claim:", safe_claim)
print("Uses observed language: YES")
print("Uses measured language: YES")
print("Uses directional language: YES")
print("Uses decision-support language: YES")
print("Causal guarantee removed: YES")
print("===================================")

ML-09 CLAIM REWRITE CHECK
Safe claim: The model provides measured, directional decision-support for prioritizing content pages for review based on observed search and engagement signals.
Uses observed language: YES
Uses measured language: YES
Uses directional language: YES
Uses decision-support language: YES
Causal guarantee removed: YES


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

In [5]:
# ============================================================
# ML-09 FINAL SELF-CHECK
# ============================================================

assert len(paper_findings) == 2
assert len(methodology_questions) == 2

assert client_overlap == 0
assert spearman_grouped == 0.8638
assert top20_overlap == 10

assert len(leakage_found) == 0
assert leaky_score > honest_score

assert "observed" in safe_claim.lower()
assert "measured" in safe_claim.lower()
assert "directional" in safe_claim.lower()
assert "decision-support" in safe_claim.lower()

print("===================================")
print("ML-09 VALIDATION AUDIT")
print("===================================")
print("Lane: Refresh / Content Opportunity Scoring")
print("Decision month: 2026-03")
print("Paper findings audited: 2")
print("Grouped validation: PASS")
print("Client overlap:", client_overlap)
print("Spearman:", spearman_grouped)
print("Top-20 overlap:", f"{top20_overlap}/{top20_total}")
print("Leakage audit: PASS")
print("Safe claim language: PASS")
print("Public-safe framing: PASS")
print("===================================")
print("ML-09 SELF-CHECK PASSED")
print("===================================")

ML-09 VALIDATION AUDIT
Lane: Refresh / Content Opportunity Scoring
Decision month: 2026-03
Paper findings audited: 2
Grouped validation: PASS
Client overlap: 0
Spearman: 0.8638
Top-20 overlap: 10/20
Leakage audit: PASS
Safe claim language: PASS
Public-safe framing: PASS
ML-09 SELF-CHECK PASSED
